# 02 — Full-joint driver + Laplace covariance (per-panel σ)

`FullJointDriver` refines **every** refined parameter at once in a
single LM, then computes the **Laplace (Cramér-Rao) covariance at the
MAP** — for free, from the same autograd Jacobian the LM used.  This
gives a per-parameter σ, including a σ on every panel's (δy, δz).

Self-contained synthetic data; runs in ~10 s.


In [1]:
import os, math, time
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')   # macOS OpenMP guard
import numpy as np
import torch

import midas_peakfit as mp
from midas_calibrate_v2.forward.panels import PanelLayout
from midas_hkls import Lattice, SpaceGroup
from midas_diffract.hkls import hkls_for_forward_model

# The package's validated synthetic generators (paper fig. 1 source).
import midas_joint_ff_calibrate.runners.run_synthetic_pilatus_joint as R
from midas_joint_ff_calibrate.loss import JointWeights, joint_residual
from midas_joint_ff_calibrate.pipelines.identifiability import fisher_block_rank
from midas_joint_ff_calibrate.pipelines.alternating import AlternatingDriver
from midas_joint_ff_calibrate.pipelines.full_joint import FullJointDriver

# ---- shrink the problem for notebook speed (production paths unchanged)
R.N_GRAINS = 24
R.N_PANELS_Y = 4
R.N_PANELS_Z = 4
R.PANEL_SIZE_Y = 150
R.PANEL_SIZE_Z = 150
R.LSD_UM = 7.0e5            # closer detector -> more panels see Au rings
R.N_RINGS = 6
R.TWO_THETA_MAX_DEG = 14.0
R.N_POWDER_PER_RING = 180

# Loss weights (1/sigma per modality so neither dominates) + gauge lambda.
W_POWDER, W_HEDM, LAMBDA_GAUGE = 1.0e4, 10.0, 1.0e6


def build_problem(seed: int = 2026):
    """Build (layout, truth, grains, ring 2theta/d, powder+HEDM obs)."""
    layout = PanelLayout.regular(R.N_PANELS_Y, R.N_PANELS_Z,
                                 R.PANEL_SIZE_Y, R.PANEL_SIZE_Z,
                                 gap_y=R.GAP_Y, gap_z=R.GAP_Z)
    truth = R.sample_truth(layout, seed=seed)
    grain_eulers, grain_pos, grain_lat = R.sample_truth_grains(seed=seed + 1)

    sg = SpaceGroup.from_number(225)                 # Fm-3m (Au)
    lat = Lattice.for_system('cubic', a=R.AU_LATTICE_A)
    _, thetas, _ = hkls_for_forward_model(
        sg, lat, wavelength_A=R.WAVELENGTH_A,
        two_theta_max_deg=R.TWO_THETA_MAX_DEG, expand_equivalents=False)
    ring_tt, _ = torch.unique(2 * thetas * 180.0 / math.pi,
                              return_inverse=True, sorted=True)
    ring_tt = ring_tt.double()[:R.N_RINGS]
    ring_d = R.WAVELENGTH_A / (2.0 * torch.sin(ring_tt * math.pi / 360.0))

    powder_obs = R.generate_powder_observations(layout, truth, ring_tt, seed=seed + 2)
    hedm_obs = R.generate_hedm_observations(
        layout, truth, grain_eulers, grain_pos, grain_lat, seed=seed + 3)
    return dict(layout=layout, truth=truth,
                grain_eulers=grain_eulers, grain_pos=grain_pos, grain_lat=grain_lat,
                ring_tt=ring_tt, ring_d=ring_d,
                powder_obs=powder_obs, hedm_obs=hedm_obs)


def build_spec(prob, *, refine_grains=False, refine_panels=True):
    """Canonical joint spec: geometry + per-panel deltas + grain blocks."""
    truth = prob['truth']; layout = prob['layout']
    spec = mp.ParameterSpec()
    spec.add(mp.Parameter('Lsd', init=truth.Lsd + 50.0,
                          bounds=(truth.Lsd - 5e3, truth.Lsd + 5e3)))
    spec.add(mp.Parameter('BC_y', init=truth.BC_y + 0.3,
                          bounds=(truth.BC_y - 5.0, truth.BC_y + 5.0)))
    spec.add(mp.Parameter('BC_z', init=truth.BC_z - 0.2,
                          bounds=(truth.BC_z - 5.0, truth.BC_z + 5.0)))
    spec.add(mp.Parameter('ty', init=0.0, refined=False))
    spec.add(mp.Parameter('tz', init=0.0, refined=False))
    spec.add(mp.Parameter('Wavelength', init=R.WAVELENGTH_A, refined=False))
    spec.add(mp.Parameter('pxY', init=R.PX_UM, refined=False))
    spec.add(mp.Parameter('pxZ', init=R.PX_UM, refined=False))
    spec.add(mp.Parameter('RhoD', init=200000.0, refined=False))
    spec.add(mp.Parameter('panel_delta_yz',
                          init=torch.zeros(layout.n_panels(), 2, dtype=torch.float64),
                          bounds=(-3.0, 3.0),
                          prior=mp.GaussianPrior(mean=0.0, std=0.5),
                          refined=refine_panels))
    spec.add(mp.Parameter('panel_delta_theta',
                          init=torch.zeros(layout.n_panels(), dtype=torch.float64),
                          refined=False))
    spec.add(mp.Parameter('grain_euler', init=prob['grain_eulers'],
                          bounds=(-2 * math.pi, 2 * math.pi), refined=refine_grains))
    spec.add(mp.Parameter('grain_pos', init=prob['grain_pos'],
                          bounds=(-1000.0, 1000.0), refined=refine_grains))
    spec.add(mp.Parameter('grain_lattice', init=prob['grain_lat'], refined=False))
    return spec


def make_closures(prob, spec):
    """Return (joint, powder_only, hedm_only) gauge-free residual closures."""
    pf = R.make_powder_residual(prob['powder_obs'], prob['layout'],
                                prob['ring_tt'], ring_d_spacing_A=prob['ring_d'])
    hf = R.make_hedm_residual(prob['hedm_obs'], prob['layout'])

    def joint_fn(u):
        return joint_residual(
            u, powder_residual_fn=pf, hedm_residual_fn=hf, spec=spec,
            weights=JointWeights(w_powder=W_POWDER, w_hedm=W_HEDM,
                                 lambda_gauge=LAMBDA_GAUGE),
            gauge_blocks=[])

    def powder_only(u):   return W_POWDER * pf(u)
    def hedm_only(u):     return W_HEDM * hf(u)
    return joint_fn, powder_only, hedm_only


def seed_unpacked(spec):
    """Dict of every parameter at its init value, as float64 tensors."""
    u = {n: spec.parameters[n].init_tensor() for n in spec.parameters}
    for n in u:
        if not isinstance(u[n], torch.Tensor):
            u[n] = torch.tensor(u[n], dtype=torch.float64)
    return u


def covered_panels(prob):
    p = set(prob['powder_obs']['panel_idx'].tolist())
    h = set(prob['hedm_obs']['panel_idx'].tolist())
    return p, h, (p | h)


DIPlib -- a quantitative image analysis library
Version 3.5.2 (Dec 27 2024)
For more information see https://diplib.org


/Users/hsharma/miniconda3/envs/midas_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
prob = build_problem()
layout = prob['layout']; truth = prob['truth']
spec = build_spec(prob, refine_grains=False, refine_panels=True)
joint_fn, powder_only, hedm_only = make_closures(prob, spec)
p_pan, h_pan, cov = covered_panels(prob)
print(f'{layout.n_panels()} panels, {len(cov)} covered; refined: {spec.refined_names()}')


16 panels, 12 covered; refined: ['Lsd', 'BC_y', 'BC_z', 'panel_delta_yz']


## Run FullJointDriver with `compute_laplace=True`

`sigma_r` is the residual noise scale (the post-weighting residual is
~unit-variance, so `sigma_r=1.0`).  The driver returns a
`LaplaceResult` with `sigma_per_dim` aligned to `refined_names`.


In [3]:
drv = FullJointDriver(
    spec=spec,
    residual_fn=joint_fn,
    lm_config=mp.GenericLMConfig(max_iter=80, ftol_rel=1e-10, xtol_rel=1e-10),
    fallback_span=2.0,
    sigma_r=1.0,
    compute_laplace=True,
)
t0 = time.time()
res = drv.run()
print(f'rc={res.rc}  cost={res.cost:.4e}  time={time.time()-t0:.1f}s')
print(f'laplace computed: {res.laplace is not None}')
lap = res.laplace
print(f'sigma_per_dim length: {lap.sigma_per_dim.numel()}  '
      f'(refined names: {lap.refined_names})')


rc=0  cost=3.0819e+03  time=6.7s
laplace computed: True
sigma_per_dim length: 35  (refined names: ['Lsd', 'BC_y', 'BC_z', 'panel_delta_yz'])


## Per-parameter σ on geometry

Read off the σ on the scalar geometry parameters.


In [4]:
names = lap.refined_names
sig = lap.sigma_per_dim
# The flat sigma vector expands vector params; map scalar names to their
# first (only) entry by walking offsets.
import numpy as np
offset = 0
for n in names:
    p = spec.parameters[n]
    size = int(np.prod(p.init.shape)) if hasattr(p.init, 'shape') and p.init.dim() else 1
    if size == 1:
        print(f'  sigma({n:<14s}) = {float(sig[offset]):.4e}')
    offset += size


  sigma(Lsd           ) = 4.9812e+01
  sigma(BC_y          ) = 3.3858e-02
  sigma(BC_z          ) = 3.3875e-02


## Per-panel σ map on `panel_delta_yz`

The headline Bayesian output: a σ for each panel's (δy, δz).  Panels
seen by data have small σ; uncovered panels saturate at the prior σ
(0.5 px).  We reshape the panel block of `sigma_per_dim` to a grid.


In [5]:
# Find the slice of the flat sigma vector belonging to panel_delta_yz.
offset = 0
panel_sigma = None
for n in names:
    p = spec.parameters[n]
    size = int(np.prod(p.init.shape)) if hasattr(p.init, 'shape') and p.init.dim() else 1
    if n == 'panel_delta_yz':
        panel_sigma = sig[offset:offset + size].reshape(layout.n_panels(), 2)
        break
    offset += size

covered_mask = torch.zeros(layout.n_panels(), dtype=torch.bool)
for k in cov:
    covered_mask[k] = True

sig_norm = panel_sigma.norm(dim=1)
print(f'{"panel":>6s}  {"covered":>8s}  {"sigma_dy":>10s}  {"sigma_dz":>10s}')
for k in range(layout.n_panels()):
    print(f'{k:>6d}  {str(bool(covered_mask[k])):>8s}  '
          f'{float(panel_sigma[k,0]):>10.4f}  {float(panel_sigma[k,1]):>10.4f}')
print(f'\nmedian sigma (covered)   = {float(sig_norm[covered_mask].median()):.4f} px')
print(f'median sigma (uncovered) = {float(sig_norm[~covered_mask].median()):.4f} px '
      f'(should approx the 0.5 px prior)')


 panel   covered    sigma_dy    sigma_dz
     0      True      0.0327      0.0327
     1      True      0.0309      0.0341
     2      True      0.0309      0.0394
     3      True      0.0313      0.0451
     4      True      0.0386      0.0315
     5     False      0.5000      0.5000
     6     False      0.5000      0.5000
     7      True      0.0382      0.0480
     8      True      0.0449      0.0315
     9     False      0.5000      0.5000
    10     False      0.5000      0.5000
    11      True      0.0446      0.0480
    12      True      0.0454      0.0313
    13      True      0.0481      0.0341
    14      True      0.0481      0.0394
    15      True      0.0453      0.0451

median sigma (covered)   = 0.0549 px
median sigma (uncovered) = 0.7071 px (should approx the 0.5 px prior)


In [6]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

grid = sig_norm.numpy().reshape(R.N_PANELS_Y, R.N_PANELS_Z).astype(float)
cov2d = covered_mask.numpy().reshape(R.N_PANELS_Y, R.N_PANELS_Z)
grid_masked = grid.copy()
grid_masked[~cov2d] = np.nan
fig, ax = plt.subplots(figsize=(5, 4.2))
im = ax.imshow(grid_masked, cmap='viridis', origin='lower')
ax.set_title('Per-panel sigma(|delta_yz|) at MAP (covered panels)')
ax.set_xlabel('panel col'); ax.set_ylabel('panel row')
fig.colorbar(im, ax=ax, label='sigma (px)')
fig.tight_layout()
fig.savefig('full_joint_panel_sigma.png', dpi=120)
plt.close(fig)
print('wrote full_joint_panel_sigma.png')


wrote full_joint_panel_sigma.png


## Takeaway

`FullJointDriver(... compute_laplace=True)` gives a calibrated σ on
every panel shift in one shot.  The next notebook (**03**) explains
*why* the joint fit is needed at all: powder alone leaves the
per-panel block rank-deficient.
